In [1]:
import sys
sys.executable

'/Users/carloskim/Developer/DE_zoomcamp/DE_zoomcamp/07_homework/.venv/bin/python'

In [2]:
import pandas as pd

In [3]:
data = 'green_tripdata_2025-10.parquet'

columns = ['lpep_pickup_datetime',
'lpep_dropoff_datetime',
'PULocationID',
'DOLocationID',
'passenger_count',
'trip_distance',
'tip_amount',
'total_amount']

df = pd.read_parquet(data, columns=columns)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [4]:
from dataclasses import dataclass

@dataclass
class Ride:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: float
    trip_distance: float
    tip_amount: float
    total_amount: float

In [5]:
def ride_from_row(row):
    return Ride(
        lpep_pickup_datetime=str(row['lpep_pickup_datetime']),
        lpep_dropoff_datetime=str(row['lpep_dropoff_datetime']),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=float(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

In [6]:
ride = ride_from_row(df.iloc[0])
ride

Ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1.0, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [7]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

In [8]:
topic_name = 'green-trips'
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [9]:
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [10]:
from time import time
import dataclasses

t0 = time()

# send all rows
for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 11.09 seconds


In [25]:
def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)
    
from kafka import KafkaConsumer

server = 'localhost:9092'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console-V2',
    value_deserializer=ride_deserializer,
    consumer_timeout_ms=5000
)

In [24]:
count = 0

for message in consumer:
    ride = message.value
    
    if ride.trip_distance > 5:
        count += 1

print("Trips with distance > 5 km:", count)

Trips with distance > 5 km: 8506


In [26]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5433,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [28]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.lpep_pickup_datetime, ride.lpep_dropoff_datetime, ride.PULocationID, ride.DOLocationID,
         ride.passenger_count, ride.trip_distance, ride.tip_amount,
         ride.total_amount)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...


TypeError: not all arguments converted during string formatting